In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q03/annotated/checkpoints/pre_cell_3.pickle

trying: ['tpch_parent']
me:  0
trying: ['orders']


me:  2
trying: ['DATA_ROOT']
me:  0
trying: ['load_lineitem']
me:  1
trying: ['STORAGE_OPTS']
me:  0
trying: ['pd']
me:  0
trying: ['lineitem']


me:  1
trying: ['load_customer']
me:  3
trying: ['customer']
me:  3
trying: ['load_orders']
me:  2


Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable orders
Declaring variable load_orders
Declaring variable load_customer
Declaring variable customer


In [4]:
%%RecordEvent
%%time
### cell 3 ###

# keep using pd.Timestamp (pd is the cudf.pandas wrapper)
date = pd.Timestamp("1995-03-04")

# project only the needed columns
lineitem_filtered = lineitem.loc[:, ["L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT", "L_SHIPDATE"]]
orders_filtered   = orders.loc[:,   ["O_ORDERKEY",  "O_CUSTKEY",   "O_ORDERDATE", "O_SHIPPRIORITY"]]
customer_filtered = customer.loc[:, ["C_MKTSEGMENT",  "C_CUSTKEY"]]

# apply the predicates
lsel = lineitem_filtered.L_SHIPDATE > date
osel = orders_filtered.O_ORDERDATE < date
csel = customer_filtered.C_MKTSEGMENT == "HOUSEHOLD"

# use direct boolean indexing (not .loc) to preserve exactly the same indexing semantics
flineitem = lineitem_filtered[lsel]
forders     = orders_filtered[osel]
fcustomer   = customer_filtered[csel]

# join customers -> orders, then join that with lineitem
# add sort=False to match pandas' default merge ordering
jn1 = fcustomer.merge(
    forders,
    left_on  = "C_CUSTKEY",
    right_on = "O_CUSTKEY",
    sort=False
)
jn2 = jn1.merge(
    flineitem,
    left_on  = "O_ORDERKEY",
    right_on = "L_ORDERKEY",
    sort=False
)

# compute revenue per line
jn2["TMP"] = jn2.L_EXTENDEDPRICE * (1 - jn2.L_DISCOUNT)

# group by the three keys, sum TMP, then sort by TMP descending
total = (
    jn2
      .groupby(
          ["L_ORDERKEY", "O_ORDERDATE", "O_SHIPPRIORITY"],
          as_index=False,
          sort=False
      )["TMP"]
      .sum()
      .sort_values("TMP", ascending=False)
)

# final projection of columns
res = total[["L_ORDERKEY", "TMP", "O_ORDERDATE", "O_SHIPPRIORITY"]]

CPU times: user 128 ms, sys: 63.9 ms, total: 191 ms
Wall time: 204 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q03/rewritten/o4_mini_high/checkpoints/post_cell_3_try_3.pickle

migration speed (bps): 807298722.14608
---------------------------
variables to migrate:
date 48
lsel 48
STORAGE_OPTS 64
customer 48
load_customer 144
orders 48
load_orders 144
load_lineitem 144
lineitem 48
fcustomer 48
orders_filtered 48
flineitem 48
customer_filtered 48
res 48
csel 48
lineitem_filtered 48
tpch_parent 54
jn2 48
DATA_ROOT 80
total 48
forders 48
pd 72
osel 48
jn1 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q03/rewritten/o4_mini_high/checkpoints/post_cell_3_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables ['lineitem', 'orders']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['forders', 'osel', 'total', 'fcustomer', 'jn1', 'date', 'jn2', 'csel', 'lsel', 'orders_filtered', 'lineitem_filtered', 'res', 'flineitem', 'customer_filtered']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q03/opt_cell_exec_info_3_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [8]:
opt_output = Out.get(4)

In [9]:
res_opt = res
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q03/annotated/checkpoints/post_cell_3.pickle
assert compare_df(res_opt, res)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['csel']
me:  4
trying: ['osel']
me:  4
trying: ['orig_output']
me:  5
trying: ['lineitem_filtered']


me:  4
trying: ['customer']
me:  3
trying: ['load_customer']
me:  3
trying: ['date']
me:  4
trying: ['total']
me:  4
trying: ['fcustomer']
me:  4
trying: ['forders']
me:  4
trying: ['orders']


me:  2
trying: ['DATA_ROOT']
me:  0
trying: ['flineitem']
me:  4
trying: ['load_orders']
me:  2
trying: ['customer_filtered']
me:  4
trying: ['jn1']
me:  4
trying: ['load_lineitem']
me:  1
trying: ['orders_filtered']
me:  4
trying: ['lineitem']


me:  1
trying: ['res']
me:  4
trying: ['pd']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['jn2']
me:  4
trying: ['lsel']
me:  4
trying: ['tpch_parent']
me:  0


Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable orders
Declaring variable load_orders
Declaring variable customer
Declaring variable load_customer
Declaring variable csel
Declaring variable osel
Declaring variable lineitem_filtered
Declaring variable date
Declaring variable total
Declaring variable fcustomer
Declaring variable forders
Declaring variable flineitem
Declaring variable customer_filtered
Declaring variable jn1
Declaring variable orders_filtered
Declaring variable res
Declaring variable jn2
Declaring variable lsel
Declaring variable orig_output


ValueError: Content of df1 and df2 do not match